# 第二十一课(案例):解剖一台真实的大语言模型

讲义:[21-real-llm-anatomy.md](21-real-llm-anatomy.md)。

解剖对象:本机 HuggingFace 缓存里的 **Qwen3-4B-Instruct-2507**(bf16)。所有 cell 读真实文件,不加载权重。

In [1]:
import json, glob, os, struct, math

HUB = '/Volumes/Sam-APFS/huggingface/hub'
MODEL = 'models--mlx-community--Qwen3-4B-Instruct-2507-bf16'
snap = glob.glob(os.path.join(HUB, MODEL, 'snapshots', '*'))[0]

assert os.path.isdir(snap), f'找不到快照目录: {snap}(若换了机器,改上面 HUB 路径)'
print('解剖对象:', MODEL)
print('快照文件:', sorted(os.listdir(snap)))

解剖对象: models--mlx-community--Qwen3-4B-Instruct-2507-bf16
快照文件: ['chat_template.jinja', 'config.json', 'generation_config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json']


## 1. config.json:一页说明书(讲义 §2)

In [2]:
cfg = json.load(open(os.path.join(snap, 'config.json')))
for k in ['hidden_size', 'num_hidden_layers', 'num_attention_heads', 'num_key_value_heads',
          'head_dim', 'intermediate_size', 'vocab_size', 'max_position_embeddings',
          'rope_theta', 'rms_norm_eps', 'tie_word_embeddings', 'hidden_act']:
    print(f'{k:26} = {cfg[k]}')
print(f'GQA 共享比 Q:KV = {cfg["num_attention_heads"]}:{cfg["num_key_value_heads"]} =',
      cfg['num_attention_heads'] // cfg['num_key_value_heads'], ': 1')
print('头数×头维 =', cfg['num_attention_heads'] * cfg['head_dim'],
      ' vs hidden_size =', cfg['hidden_size'], ' ← 可以不相等!')

hidden_size                = 2560
num_hidden_layers          = 36
num_attention_heads        = 32
num_key_value_heads        = 8
head_dim                   = 128
intermediate_size          = 9728
vocab_size                 = 151936
max_position_embeddings    = 262144
rope_theta                 = 5000000
rms_norm_eps               = 1e-06
tie_word_embeddings        = True
hidden_act                 = silu
GQA 共享比 Q:KV = 32:8 = 4 : 1
头数×头维 = 4096  vs hidden_size = 2560  ← 可以不相等!


## 2. safetensors 解剖:读文件头,不加载权重(讲义 §3)

In [3]:
def read_safetensors_header(path):
    with open(path, 'rb') as f:
        n = struct.unpack('<Q', f.read(8))[0]        # 前 8 字节: 头部长度
        return json.loads(f.read(n))                 # 随后: JSON 张量表

tensors = {}
for st in sorted(glob.glob(snap + '/*.safetensors')):
    for name, meta in read_safetensors_header(st).items():
        if name != '__metadata__':
            tensors[name] = (meta['shape'], meta['dtype'])

print('张量总数:', len(tensors))
print('\n第 0 层真实张量:')
for name in sorted(tensors):
    if '.0.' in name:
        print(f'  {name:58} {str(tensors[name][0]):18} {tensors[name][1]}')

张量总数: 398

第 0 层真实张量:
  model.layers.0.input_layernorm.weight                      [2560]             BF16
  model.layers.0.mlp.down_proj.weight                        [2560, 9728]       BF16
  model.layers.0.mlp.gate_proj.weight                        [9728, 2560]       BF16
  model.layers.0.mlp.up_proj.weight                          [9728, 2560]       BF16
  model.layers.0.post_attention_layernorm.weight             [2560]             BF16
  model.layers.0.self_attn.k_norm.weight                     [128]              BF16
  model.layers.0.self_attn.k_proj.weight                     [1024, 2560]       BF16
  model.layers.0.self_attn.o_proj.weight                     [2560, 4096]       BF16
  model.layers.0.self_attn.q_norm.weight                     [128]              BF16
  model.layers.0.self_attn.q_proj.weight                     [4096, 2560]       BF16
  model.layers.0.self_attn.v_proj.weight                     [1024, 2560]       BF16


In [4]:
# 参数总账:逐张量求和 vs 按课内公式分账
by_tensor = sum(math.prod(s) for s, _ in tensors.values())

d = cfg['hidden_size']; h = cfg['intermediate_size']
q_dim = cfg['num_attention_heads'] * cfg['head_dim']     # 4096
kv_dim = cfg['num_key_value_heads'] * cfg['head_dim']    # 1024
emb = cfg['vocab_size'] * d
attn = q_dim*d + 2*kv_dim*d + d*q_dim
ffn = 3 * d * h
norms = cfg['num_hidden_layers'] * (2*d + 2*cfg['head_dim']) + d
by_formula = emb + cfg['num_hidden_layers'] * (attn + ffn) + norms

print(f'逐张量求和   : {by_tensor:,}')
print(f'公式分账     : {by_formula:,}')
print('对账一致     :', by_tensor == by_formula)
print(f'bf16 大小    : {by_tensor*2/1024**3:.2f} GB')
print(f'  其中 词嵌入 {emb:,} | 每层注意力 {attn:,} | 每层FFN {ffn:,}')

逐张量求和   : 4,022,468,096
公式分账     : 4,022,468,096
对账一致     : True
bf16 大小    : 7.49 GB
  其中 词嵌入 388,956,160 | 每层注意力 26,214,400 | 每层FFN 74,711,040


## 3. generation_config.json + KV Cache 真实账(讲义 §5–6)

In [5]:
gen = json.load(open(os.path.join(snap, 'generation_config.json')))
print('真实采样默认:', {k: gen[k] for k in ('do_sample', 'temperature', 'top_k', 'top_p')})

def kv_cache_gb(cfg, seq, batch=1, bytes_per=2):
    return 2 * cfg['num_hidden_layers'] * cfg['num_key_value_heads'] * cfg['head_dim'] * seq * batch * bytes_per / 1024**3

for seq in (8192, 32768, 262144):
    print(f'KV Cache @{seq//1024:>4}K batch=1: {kv_cache_gb(cfg, seq):6.2f} GB')

真实采样默认: {'do_sample': True, 'temperature': 0.7, 'top_k': 20, 'top_p': 0.8}
KV Cache @   8K batch=1:   1.12 GB
KV Cache @  32K batch=1:   4.50 GB
KV Cache @ 256K batch=1:  36.00 GB


## 4. tokenizer:词表与 <think>(讲义 §7)

In [7]:
tok = json.load(open(os.path.join(snap, 'tokenizer.json')))
vocab = tok['model']['vocab']
added = {t['content']: t['id'] for t in tok['added_tokens']}
print('基础词表:', len(vocab), ' 特殊 token:', len(added),
      ' 容量 vocab_size:', cfg['vocab_size'], f'(=对齐到 128 倍数: {cfg["vocab_size"] % 128 == 0})')
for s in ['<|endoftext|>', '<|im_start|>', '<|im_end|>', '<think>', '</think>']:
    print(f'  {s:16} id={added.get(s)}')
print("'attention' → id", vocab.get('attention'), '(一个词一个 id,第2课查表)')

基础词表: 151643  特殊 token: 26  容量 vocab_size: 151936 (=对齐到 128 倍数: True)
  <|endoftext|>    id=151643
  <|im_start|>     id=151644
  <|im_end|>       id=151645
  <think>          id=151667
  </think>         id=151668
'attention' → id 53103 (一个词一个 id,第2课查表)


## 作业现场

- C4:遍历本地全部模型做对照表(下方)。
- D1:对 Qwen3-1.7B / Qwen2.5-1.5B / Llama-3.2-1B 做参数对账。
- D3:多 eos 终止条件的 generate 改造。

In [6]:
# TODO(C4): 本地模型对照表
# for m in sorted(os.listdir(HUB)):
#     if not m.startswith('models--'): continue
#     paths = glob.glob(os.path.join(HUB, m, 'snapshots', '*', 'config.json'))
#     if not paths: continue
#     c = json.load(open(paths[0]))
#     print(f"{m.split('--')[-1][:34]:36} L={c.get('num_hidden_layers','-'):>3} d={c.get('hidden_size','-'):>5} "
#           f"Q/KV={c.get('num_attention_heads','-')}/{c.get('num_key_value_heads', c.get('num_attention_heads','-'))}")

# TODO(D3): 多 eos 终止
def generate_multi_eos(sample_fn, prompt, max_n, eos_ids=(151645, 151643)):
    # 你的代码:生成循环中,采到任一 eos 即停止(第14课 generate 的生产版)
    ...